In [1]:
%pip install nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# %% 1) นำเข้า library
import numpy as np
import pandas as pd
import plotly.io as pio
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# แสดงกราฟใน Jupyter ต้องมี nbformat (ใน notebook ใช้: %pip install nbformat แล้วรีสตาร์ต kernel)
# ถ้ายังไม่มี ให้เปิดกราฟในเบราว์เซอร์แทน จะได้ไม่ขึ้น ValueError
try:
    import nbformat
except ImportError:
    pio.renderers.default = "browser"

# %% 2) ดึงข้อมูลและสำรวจ (ครั้งที่ 6)
df = pd.read_excel(r"c:\Project1\shopping_trends.csv.xlsx")   # ชื่อไฟล์ต้องตรงกับที่อยู่ในโฟลเดอร์ (แก้ได้)

print(df.shape)                      # (แถว, คอลัมน์)
print(df.head())                     # 5 แถวแรก
df.info()                            # ชนิดข้อมูล
print(df.describe())                 # สถิติพื้นฐาน
print("ค่าว่างทั้งหมด:", df.isnull().sum().sum())

# สร้างช่วงอายุไว้ใช้ทำกราฟ
df["Age Group"] = pd.cut(df["Age"], [17, 25, 35, 45, 55, 70],
                         labels=["18-25", "26-35", "36-45", "46-55", "56-70"])

# ตั้งค่าสีและฟอนต์ไทย (ครั้งที่ 7)
colors = ["#4FA3D1", "#F4769B", "#8E8CD8", "#3DBE9B", "#FFA862"]
thai_font = dict(family="Sarabun, Tahoma", size=14)

# %% 3) กราฟสำรวจ: สัดส่วนหมวดสินค้า (px.pie แบบโดนัท)
fig = px.pie(df, names="Category", hole=0.42, color_discrete_sequence=colors)
fig.update_layout(title="สัดส่วนรายการซื้อแยกตามหมวดสินค้า",
                  font=thai_font, template="plotly_white")
fig.show()

# %% 4) กราฟสำรวจ: ยอดซื้อเฉลี่ยตามฤดูกาล (px.bar)
season = df.groupby("Season")["Purchase Amount (USD)"].mean().reset_index()

fig = px.bar(season, x="Season", y="Purchase Amount (USD)", color="Season",
             text_auto=".1f", color_discrete_sequence=colors,
             category_orders={"Season": ["Spring", "Summer", "Fall", "Winter"]})
fig.update_layout(title="ยอดซื้อเฉลี่ยตามฤดูกาล", showlegend=False,
                  xaxis_title="ฤดูกาล", yaxis_title="ยอดซื้อเฉลี่ย (USD)",
                  font=thai_font, template="plotly_white")
fig.show()

# %% 5) กราฟสำรวจ: การกระจายของยอดซื้อตามช่วงอายุ (px.box)
fig = px.box(df, x="Age Group", y="Purchase Amount (USD)", color="Age Group",
             color_discrete_sequence=colors,
             category_orders={"Age Group": ["18-25", "26-35", "36-45", "46-55", "56-70"]})
fig.update_layout(title="การกระจายของยอดซื้อตามช่วงอายุ", showlegend=False,
                  xaxis_title="ช่วงอายุ (ปี)", yaxis_title="ยอดซื้อ (USD)",
                  font=thai_font, template="plotly_white")
fig.show()

# %% 6) กราฟสำรวจ: แผนภาพความร้อนของสหสัมพันธ์ (px.imshow)
cols = ["Age", "Purchase Amount (USD)", "Review Rating", "Previous Purchases"]
corr = df[cols].corr()
print(corr.round(3))

fig = px.imshow(corr, text_auto=".2f",
                color_continuous_scale=["#E8456F", "#FFF1DC", "#2F8FD8"],
                zmin=-1, zmax=1)
fig.update_layout(title="สหสัมพันธ์ระหว่างตัวแปรตัวเลข", font=thai_font)
fig.show()

# %% 7) ข้อ 1: อายุ (Age) กับ ยอดซื้อ (Purchase Amount)
# กราฟกระจายพร้อมเส้นแนวโน้ม (ต้องติดตั้ง statsmodels)
fig = px.scatter(df, x="Age", y="Purchase Amount (USD)", opacity=0.5,
                 trendline="ols", color_discrete_sequence=[colors[0]])
fig.update_layout(title="อายุกับยอดซื้อ",
                  xaxis_title="อายุ (ปี)", yaxis_title="ยอดซื้อ (USD)",
                  font=thai_font, template="plotly_white")
fig.show()

# เตรียม X (2 มิติ) และ y (1 มิติ) แล้วแบ่ง train / test
X = df[["Age"]]
y = df["Purchase Amount (USD)"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# สร้างโมเดลและสอน (fit เฉพาะ train)
model = LinearRegression()
model.fit(X_tr, y_tr)
print("coef:", np.round(model.coef_, 4), " intercept:", round(model.intercept_, 4))

# ประเมินผลด้วยชุด test
y_hat = model.predict(X_te)
r2_train_1 = model.score(X_tr, y_tr)
r2_test_1 = r2_score(y_te, y_hat)
rmse_1 = np.sqrt(mean_squared_error(y_te, y_hat))
print("R² train:", round(r2_train_1, 4))
print("R² test :", round(r2_test_1, 4))
print("RMSE    :", round(rmse_1, 4))
print("MAE     :", round(mean_absolute_error(y_te, y_hat), 4))

# residual plot
fig = px.scatter(x=y_hat, y=y_te - y_hat, opacity=0.5,
                 color_discrete_sequence=[colors[2]])
fig.add_hline(y=0)
fig.update_layout(title="Residual plot: อายุ → ยอดซื้อ",
                  xaxis_title="ค่าพยากรณ์", yaxis_title="ส่วนเหลือ (y - ŷ)",
                  font=thai_font, template="plotly_white")
fig.show()

# ลองเส้นโค้ง degree 2 ด้วย Pipeline
pipe = make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                     LinearRegression())
pipe.fit(X_tr, y_tr)
r2_poly_1 = pipe.score(X_te, y_te)
print("R² test degree 2:", round(r2_poly_1, 4))

# %% 8) ข้อ 2: จำนวนครั้งที่เคยซื้อ (Previous Purchases) กับ คะแนนรีวิว (Review Rating)
fig = px.scatter(df, x="Previous Purchases", y="Review Rating", opacity=0.5,
                 trendline="ols", color_discrete_sequence=[colors[3]])
fig.update_layout(title="จำนวนครั้งที่เคยซื้อกับคะแนนรีวิว",
                  xaxis_title="จำนวนครั้งที่เคยซื้อ", yaxis_title="คะแนนรีวิว",
                  font=thai_font, template="plotly_white")
fig.show()

X = df[["Previous Purchases"]]
y = df["Review Rating"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_tr, y_tr)
print("coef:", np.round(model.coef_, 4), " intercept:", round(model.intercept_, 4))

y_hat = model.predict(X_te)
r2_train_2 = model.score(X_tr, y_tr)
r2_test_2 = r2_score(y_te, y_hat)
rmse_2 = np.sqrt(mean_squared_error(y_te, y_hat))
print("R² train:", round(r2_train_2, 4))
print("R² test :", round(r2_test_2, 4))
print("RMSE    :", round(rmse_2, 4))
print("MAE     :", round(mean_absolute_error(y_te, y_hat), 4))

fig = px.scatter(x=y_hat, y=y_te - y_hat, opacity=0.5,
                 color_discrete_sequence=[colors[2]])
fig.add_hline(y=0)
fig.update_layout(title="Residual plot: จำนวนครั้งที่เคยซื้อ → คะแนนรีวิว",
                  xaxis_title="ค่าพยากรณ์", yaxis_title="ส่วนเหลือ (y - ŷ)",
                  font=thai_font, template="plotly_white")
fig.show()

pipe = make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                     LinearRegression())
pipe.fit(X_tr, y_tr)
r2_poly_2 = pipe.score(X_te, y_te)
print("R² test degree 2:", round(r2_poly_2, 4))

# %% 9) ข้อ 3: ยอดซื้อ (Purchase Amount) กับ คะแนนรีวิว (Review Rating)
fig = px.scatter(df, x="Purchase Amount (USD)", y="Review Rating", opacity=0.5,
                 trendline="ols", color_discrete_sequence=[colors[1]])
fig.update_layout(title="ยอดซื้อกับคะแนนรีวิว",
                  xaxis_title="ยอดซื้อ (USD)", yaxis_title="คะแนนรีวิว",
                  font=thai_font, template="plotly_white")
fig.show()

X = df[["Purchase Amount (USD)"]]
y = df["Review Rating"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_tr, y_tr)
print("coef:", np.round(model.coef_, 4), " intercept:", round(model.intercept_, 4))

y_hat = model.predict(X_te)
r2_train_3 = model.score(X_tr, y_tr)
r2_test_3 = r2_score(y_te, y_hat)
rmse_3 = np.sqrt(mean_squared_error(y_te, y_hat))
print("R² train:", round(r2_train_3, 4))
print("R² test :", round(r2_test_3, 4))
print("RMSE    :", round(rmse_3, 4))
print("MAE     :", round(mean_absolute_error(y_te, y_hat), 4))

fig = px.scatter(x=y_hat, y=y_te - y_hat, opacity=0.5,
                 color_discrete_sequence=[colors[2]])
fig.add_hline(y=0)
fig.update_layout(title="Residual plot: ยอดซื้อ → คะแนนรีวิว",
                  xaxis_title="ค่าพยากรณ์", yaxis_title="ส่วนเหลือ (y - ŷ)",
                  font=thai_font, template="plotly_white")
fig.show()

pipe = make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                     LinearRegression())
pipe.fit(X_tr, y_tr)
r2_poly_3 = pipe.score(X_te, y_te)
print("R² test degree 2:", round(r2_poly_3, 4))

# %% 10) สรุปผลทั้ง 3 ข้อ
summary = pd.DataFrame({
    "X → y": ["Age → Purchase Amount",
              "Previous Purchases → Review Rating",
              "Purchase Amount → Review Rating"],
    "R² train": [r2_train_1, r2_train_2, r2_train_3],
    "R² test": [r2_test_1, r2_test_2, r2_test_3],
    "RMSE test": [rmse_1, rmse_2, rmse_3],
    "R² test (degree 2)": [r2_poly_1, r2_poly_2, r2_poly_3],
})
print(summary.round(4).to_string(index=False))
# สรุป: R² ใกล้ 0 ทุกคู่ และเส้นโค้งไม่ช่วย -> ไม่พบความสัมพันธ์เชิงเส้น

(3900, 19)
   Customer ID  Age Gender Item Purchased  Category  Purchase Amount (USD)  \
0            1   55   Male         Blouse  Clothing                     53   
1            2   19   Male        Sweater  Clothing                     64   
2            3   50   Male          Jeans  Clothing                     73   
3            4   21   Male        Sandals  Footwear                     90   
4            5   45   Male         Blouse  Clothing                     49   

        Location Size      Color  Season  Review Rating Subscription Status  \
0       Kentucky    L       Gray  Winter            3.1                 Yes   
1          Maine    L     Maroon  Winter            3.1                 Yes   
2  Massachusetts    S     Maroon  Spring            3.1                 Yes   
3   Rhode Island    M     Maroon  Spring            3.5                 Yes   
4         Oregon    M  Turquoise  Spring            2.7                 Yes   

  Payment Method  Shipping Type Discount Appl

                         Age  Purchase Amount (USD)  Review Rating  \
Age                    1.000                 -0.010         -0.022   
Purchase Amount (USD) -0.010                  1.000          0.031   
Review Rating         -0.022                  0.031          1.000   
Previous Purchases     0.040                  0.008          0.004   

                       Previous Purchases  
Age                                 0.040  
Purchase Amount (USD)               0.008  
Review Rating                       0.004  
Previous Purchases                  1.000  


coef: [-0.0213]  intercept: 61.0816
R² train: 0.0002
R² test : -0.0067
RMSE    : 23.7341
MAE     : 20.7524


R² test degree 2: -0.0066


coef: [0.0009]  intercept: 3.7278
R² train: 0.0003
R² test : -0.0021
RMSE    : 0.7392
MAE     : 0.6487


R² test degree 2: -0.0019


coef: [0.0008]  intercept: 3.7032
R² train: 0.0007
R² test : 0.0018
RMSE    : 0.7378
MAE     : 0.647


R² test degree 2: 0.0013
                             X → y  R² train  R² test  RMSE test  R² test (degree 2)
             Age → Purchase Amount    0.0002  -0.0067    23.7341             -0.0066
Previous Purchases → Review Rating    0.0003  -0.0021     0.7392             -0.0019
   Purchase Amount → Review Rating    0.0007   0.0018     0.7378              0.0013


In [3]:
# %% ข้อ 2: คะแนนรีวิวแยกตามจำนวนครั้งที่เคยซื้อ (px.box)
df["Prev Group"] = pd.cut(df["Previous Purchases"], [0, 10, 20, 30, 40, 50],
                          labels=["1-10", "11-20", "21-30", "31-40", "41-50"])

fig = px.box(df, x="Prev Group", y="Review Rating", color="Prev Group",
             color_discrete_sequence=colors,
             category_orders={"Prev Group": ["1-10", "11-20", "21-30", "31-40", "41-50"]})
fig.update_layout(title="คะแนนรีวิวแยกตามจำนวนครั้งที่เคยซื้อ",
                  xaxis_title="จำนวนครั้งที่เคยซื้อ", yaxis_title="คะแนนรีวิว",
                  font=thai_font, template="plotly_white")
fig.show()

# %% ข้อ 3: คะแนนรีวิวแยกตามช่วงยอดซื้อ (px.box)
df["Amount Group"] = pd.cut(df["Purchase Amount (USD)"], [19, 40, 60, 80, 100],
                            labels=["$20-40", "$41-60", "$61-80", "$81-100"])

fig = px.box(df, x="Amount Group", y="Review Rating", color="Amount Group",
             color_discrete_sequence=colors,
             category_orders={"Amount Group": ["$20-40", "$41-60", "$61-80", "$81-100"]})
fig.update_layout(title="คะแนนรีวิวแยกตามช่วงยอดซื้อ",
                  xaxis_title="ช่วงยอดซื้อ (USD)", yaxis_title="คะแนนรีวิว",
                  font=thai_font, template="plotly_white")